<a href="https://colab.research.google.com/github/uppikaur/aml/blob/master/RNNAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Dataset
"""
data = "hello world this is a simple rnn example to be or not to be that is the question learning rnn is interesting and useful for sequence prediction"

In [ ]:
"""
Character Mapping
"""
chars = list(set(data))
data_size, vocab_size = len(data), len(chars)

char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

In [ ]:
"""
Hyperparameters
"""
hidden_size = 50
seq_length = 10
learning_rate = 0.01

In [ ]:
"""
Initialize Weights
"""
import numpy as np

Wxh = np.random.randn(hidden_size, vocab_size) * 0.01  # input → hidden
Whh = np.random.randn(hidden_size, hidden_size) * 0.01 # hidden → hidden
Why = np.random.randn(vocab_size, hidden_size) * 0.01  # hidden → output

bh = np.zeros((hidden_size, 1))
by = np.zeros((vocab_size, 1))

In [ ]:
"""
Helper Functions
"""
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / np.sum(e_x)

In [ ]:
"""
Forward Propagation Through Time
"""
def forward(inputs, targets, hprev):
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(hprev)
    loss = 0

    for t in range(len(inputs)):
        xs[t] = np.zeros((vocab_size,1))
        xs[t][inputs[t]] = 1

        hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh)

        ys[t] = np.dot(Why, hs[t]) + by
        ps[t] = softmax(ys[t])

        loss += -np.log(ps[t][targets[t],0])

    return loss, xs, hs, ps

In [ ]:
"""
Backpropagation Through Time
"""
def backward(xs, hs, ps, targets):
    dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)
    dhnext = np.zeros_like(hs[0])

    for t in reversed(range(len(xs))):
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1

        dWhy += np.dot(dy, hs[t].T)
        dby += dy

        dh = np.dot(Why.T, dy) + dhnext
        dhraw = (1 - hs[t] * hs[t]) * dh

        dbh += dhraw
        dWxh += np.dot(dhraw, xs[t].T)
        dWhh += np.dot(dhraw, hs[t-1].T)

        dhnext = np.dot(Whh.T, dhraw)

    # Gradient clipping (important!)
    for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(dparam, -5, 5, out=dparam)

    return dWxh, dWhh, dWhy, dbh, dby

In [ ]:
"""
Update Weights GD
"""
def update(dWxh, dWhh, dWhy, dbh, dby):
    global Wxh, Whh, Why, bh, by

    Wxh -= learning_rate * dWxh
    Whh -= learning_rate * dWhh
    Why -= learning_rate * dWhy
    bh  -= learning_rate * dbh
    by  -= learning_rate * dby

In [ ]:
"""
Training loop
"""
n, p = 0, 0
hprev = np.zeros((hidden_size,1))

for epoch in range(1000):
    if p + seq_length + 1 >= len(data):
        hprev = np.zeros((hidden_size,1))
        p = 0

    inputs = [char_to_ix[ch] for ch in data[p:p+seq_length]]
    targets = [char_to_ix[ch] for ch in data[p+1:p+seq_length+1]]

    loss, xs, hs, ps = forward(inputs, targets, hprev)
    dWxh, dWhh, dWhy, dbh, dby = backward(xs, hs, ps, targets)

    update(dWxh, dWhh, dWhy, dbh, dby)

    hprev = hs[seq_length-1]
    p += seq_length

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

In [ ]:
"""
Text Generation
"""
def sample(h, seed_ix, n):
    x = np.zeros((vocab_size,1))
    x[seed_ix] = 1

    ixes = []

    for t in range(n):
        h = np.tanh(np.dot(Wxh, x) + np.dot(Whh, h) + bh)
        y = np.dot(Why, h) + by
        p = softmax(y)

        ix = np.random.choice(range(vocab_size), p=p.ravel())
        x = np.zeros((vocab_size,1))
        x[ix] = 1
        ixes.append(ix)

    return ''.join(ix_to_char[ix] for ix in ixes)

In [ ]:
"""
Output
"""
print(sample(hprev, char_to_ix['h'], 100))